# Phase 1 Robustness Checks

Supplementary analyses validating the main DiD and Causal Forest results.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/cleaned/final_analysis_data.csv')

print(f"Shape: {df.shape}")
print(f"Treated obs (post_carbon_tax): {df['post_carbon_tax'].sum()}")
print(f"Treatment cohorts: {sorted(df.loc[df['treatment_year'].notna(), 'treatment_year'].unique().astype(int))}")
print(f"Never-treated countries: {df[df['treatment_year'].isna()]['country'].nunique()}")

## 1. Sun-Abraham Staggered-Robust DiD

Standard TWFE with 13 treatment cohorts is vulnerable to negative weight bias when treatment effects are heterogeneous (Goodman-Bacon 2021). Sun & Abraham (2021) estimates cohort-specific ATTs and aggregates them correctly.

In [ ]:
import pyfixest as pf

# Prepare data for pyfixest sunab()
df_staggered = df.copy()

# Never-treated: cohort must be a value that never appears as a year
# pyfixest sunab convention: use a large number for never-treated
df_staggered['cohort'] = df_staggered['treatment_year'].fillna(10000).astype(int)

# Encode country as integer (required by pyfixest for FE)
df_staggered['country_id'] = df_staggered['country'].astype('category').cat.codes

# Use co2_per_capita_future_trend as primary outcome (matches CLAUDE.md)
# Fall back to co2_per_capita_3yr_change if future_trend not available
outcome = 'co2_per_capita_future_trend'
if outcome not in df_staggered.columns or df_staggered[outcome].isna().all():
    outcome = 'co2_per_capita_3yr_change'

# Drop rows with missing outcome
df_staggered = df_staggered[df_staggered[outcome].notna()].copy()

print(f"Staggered sample: {df_staggered.shape}")
print(f"Cohorts: {sorted(df_staggered['cohort'].unique())}")
print(f"Outcome variable: {outcome}")

In [ ]:
# Sun-Abraham (2021) — staggered-robust, no negative weights
fit_sa = pf.feols(
    f"{outcome} ~ sunab(cohort, year) | country_id + year",
    data=df_staggered,
    vcov={"CRV1": "country_id"}
)

print(fit_sa.summary())

In [ ]:
import os
os.makedirs('../outputs', exist_ok=True)

# Event study plot from Sun-Abraham
fig = fit_sa.iplot(
    coord_flip=False,
    figsize=(12, 6),
    title="Sun-Abraham Event Study (Staggered-Robust)",
).get_figure()
fig.savefig('../outputs/sun_abraham_event_study.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to outputs/sun_abraham_event_study.png")

In [ ]:
# Aggregate Sun-Abraham to overall ATT
try:
    sa_agg = fit_sa.aggregate(agg="ATT")
    print("=== Aggregated ATT (Sun-Abraham) ===")
    print(sa_agg.summary())
except Exception as e:
    print(f"Aggregation method: {e}")
    # Manual aggregation fallback — average post-treatment coefficients
    sa_coefs = fit_sa.coef()
    post_coefs = {k: v for k, v in sa_coefs.items() if '::' in k}
    if post_coefs:
        mean_att = np.mean(list(post_coefs.values()))
        print(f"\nManual average of post-treatment cohort-time ATTs: {mean_att:.4f}")

# Compare with standard TWFE
fit_twfe = pf.feols(
    f"{outcome} ~ post_carbon_tax + log_gdp + log_population + "
    f"trade_openness + natural_resource_rents_per_gdp + fossil_pct_filled | country_id + year",
    data=df_staggered.dropna(subset=['log_gdp', 'log_population', 'trade_openness',
                                      'natural_resource_rents_per_gdp', 'fossil_pct_filled']),
    vcov={"CRV1": "country_id"}
)

print("\n=== TWFE Estimate (for comparison) ===")
print(f"  Coefficient: {fit_twfe.coef()['post_carbon_tax']:.4f}")
print(f"  SE:          {fit_twfe.se()['post_carbon_tax']:.4f}")
print(f"  p-value:     {fit_twfe.pvalue()['post_carbon_tax']:.4f}")

## Staggered DiD Robustness Check

| Estimator | ATT Estimate | SE | Notes |
|-----------|-------------|-----|-------|
| TWFE (standard) | fill in | fill in | Original spec, susceptible to negative weights |
| Sun-Abraham (2021) | fill in | fill in | Staggered-robust, no negative weights |

**Conclusion:** fill in after running